# Tier 1 — noise floor runs

**Purpose: measure how much `val_loss` moves between runs that differ only by random seed.**

This notebook trains the *same* configuration three times, changing nothing but
`training.seed` (42, 43, 44). The spread across those three runs is the noise floor: any
difference between two fine-tuning parameter sets that is *smaller* than this spread is
not evidence of anything. Run this before comparing pooling strategies, LoRA-vs-probe, or
dropout values — otherwise those comparisons cannot be interpreted.

It is deliberately a near-copy of **`2_finetune_template_1D.ipynb`**, so you can diff the
two. Three differences, each commented where it appears:

1. `L.seed_everything()` is called **before** the model is built, not after (cell "Run one
   seed"). In the reference notebook it comes last, which means the head initialization
   there does *not* depend on `training.seed`. For this experiment it must.
2. Model construction, LoRA, and `trainer.fit()` live inside a function so they can be
   repeated from scratch per seed.
3. The dataloaders are rebuilt per seed (`seed=` argument) so the shuffle order varies
   too, and the measured spread covers the whole run, not just weight initialization.

With 8 training and 3 validation filaments, remember what the trivial predictors score:

| Predictor | train MSE | val MSE |
|---|---|---|
| constant 0 (always dextral) | 0.1250 | 0.3333 |
| constant 0.125 (the training prior) | 0.1094 | **0.2656** |

A `val_loss` sitting near 0.2656 means the model learned the class prior and nothing about
chirality. Check that before reading anything into the seed spread.

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [24]:
import os
# This machine exposes a single L40S (44 GiB) as device 0 — `nvidia-smi -L` to confirm on
# yours. Setting a device index that does not exist leaves torch with zero visible GPUs and
# Lightning's accelerator="auto" silently falls back to CPU, which at 4096x4096 x 13
# channels will never finish.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [25]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import gc
import sys
from pathlib import Path

import pandas as pd
import torch
import wandb
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from torch.utils.data import DataLoader

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
torch.set_float32_matmul_precision('medium')


## Load configuration

Same config file as `2_finetune_template_1D.ipynb` — this notebook does not define its own
hyperparameters. Everything except the seed comes from `configs/config_script.yaml`, so the
Tier 1 measurement describes the configuration you will actually be varying later.

In [26]:
# The config is the single source of truth. load_filament_config() parses it into a typed
# object, exactly as the training script does, so the same YAML behaves identically here.
from downstream_apps.filament_kyle.configs import load_filament_config

cfg = load_filament_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: filament_characterization


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [27]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


## Pin the configuration under test

The whole point of a noise-floor measurement is that *only* the seed varies. This cell
asserts that the config on disk still matches the Tier 1 specification, so an unrelated
edit to `config_script.yaml` cannot silently change what this notebook measured.

`target_modules` is checked too: HuggingFace-style names (`q_proj`, `k_proj`, `v_proj`,
`out_proj`) match nothing in Surya, and PEFT only errors when *no* entry matches — so a
wrong list silently adapts the MLPs only (2,666,241 trainable instead of 3,157,761).

In [28]:
# Expected Tier 1 configuration. Keep this table and the YAML in agreement.
EXPECTED = {
    "model.pooling":                  ("class_token", cfg.model.pooling),
    "model.penultimate_linear_layer": (True,          cfg.model.penultimate_linear_layer),
    "model.dropout":                  (0.2,           cfg.model.dropout),
    "model.use_lora":                 (True,          cfg.model.use_lora),
    "model.freeze_backbone":          (False,         cfg.model.freeze_backbone),
    "model.lora_config.r":            (8,             cfg.model.lora_config.r),
    "model.lora_config.target_modules": (
        ["fc1", "fc2", "attn.qkv", "attn.proj"], cfg.model.lora_config.target_modules,
    ),
    "training.learning_rate":         (0.0001,        cfg.learning_rate),
    "training.batch_size":            (1,             cfg.batch_size),
    # "warn" is what makes a single run reproducible. With false, two identical runs can
    # differ and the number this notebook produces is not a seed effect at all.
    "training.deterministic":         ("warn",        cfg.deterministic),
}

bad = []
for key, (want, got) in EXPECTED.items():
    ok = want == got
    print(f"  {'OK ' if ok else 'BAD'}  {key:38s} = {got!r}" + ("" if ok else f"   (expected {want!r})"))
    if not ok:
        bad.append(f"{key}: expected {want!r}, config has {got!r}")

assert not bad, "config does not match the Tier 1 spec:\n  " + "\n  ".join(bad)
print("\nConfig matches the Tier 1 specification.")


  OK   model.pooling                          = 'class_token'
  OK   model.penultimate_linear_layer         = True
  OK   model.dropout                          = 0.2
  OK   model.use_lora                         = True
  OK   model.freeze_backbone                  = False
  OK   model.lora_config.r                    = 8
  OK   model.lora_config.target_modules       = ['fc1', 'fc2', 'attn.qkv', 'attn.proj']
  OK   training.learning_rate                 = 0.0001
  OK   training.batch_size                    = 1
  OK   training.deterministic                 = 'warn'

Config matches the Tier 1 specification.


## Run settings

`SMOKE_TEST = True` runs 2 epochs per seed so you can validate the whole loop in a few
minutes before committing to the real measurement. **A 2-epoch spread is not the noise
floor** — set `SMOKE_TEST = False` for the 20 epochs the config asks for, then re-run.

Rough cost: 8 train + 3 val samples per epoch at 4096x4096 x 13 channels. Expect very
roughly 20-40 min per 20-epoch run with a warm S3 cache, so about 1-2 hours for all three.
The first run also populates `data.s3_cache_dir` (~1 GB per unique timestep, 11 timesteps
in this catalog) if the reference notebook has not already done so.

**WandB**: this cell resolves logging up front rather than letting it fail inside
`trainer.fit()`. With no API key on the machine it switches to `WANDB_MODE=offline` — runs
are recorded locally and can be uploaded later with `wandb sync wandb/offline-run-*`. To log
online, run `wandb login` in a **terminal** (a notebook cell cannot answer the API-key
prompt, which is what raises `StdinNotImplementedError`) and re-run this cell. Setting
`USE_WANDB = False` skips WandB entirely; the CSV logger runs either way, so
`runs/noise_floor_summary.csv` is written regardless.

In [29]:
# SEEDS = [42, 43, 44]
SEEDS = [42, 43]

SMOKE_TEST = True                     # <-- set False for the real measurement
MAX_EPOCHS = 5 if SMOKE_TEST else cfg.max_epochs
USE_WANDB = False                     # <-- set True once `wandb login` has been run in a terminal

# Resolve wandb BEFORE any GPU time is spent. Lightning creates the wandb run lazily, from
# inside trainer.fit(), and with no API key on the machine wandb.init() falls back to an
# interactive login prompt. A Jupyter kernel cannot answer a stdin prompt, so the run dies
# with StdinNotImplementedError after the model is already built -- a confusing place to
# discover a credentials problem, and worse once the runs are 20 epochs long. Decide here:
#
#   USE_WANDB = False -> no wandb at all. The CSV logger still records every metric, and
#                        runs/noise_floor_summary.csv is still written, so the noise-floor
#                        measurement is unaffected. Only the browser dashboard is missing.
#   key present       -> normal online logging.
#   no key            -> WANDB_MODE=offline. Runs are written under ./wandb and never touch
#                        the network; upload later with `wandb sync wandb/offline-run-*`.
#
# wandb.api.api_key is None when no key is configured, and reading it never prompts.
if USE_WANDB:
    # Also silences wandb's "Failed to detect the name of this notebook" error.
    os.environ.setdefault("WANDB_NOTEBOOK_NAME", "noise_floor_runs.ipynb")
    if wandb.api.api_key is None:
        os.environ["WANDB_MODE"] = "offline"
        print("wandb: no API key found -> WANDB_MODE=offline "
              "(sync later with `wandb sync wandb/offline-run-*`)")
    else:
        print(f"wandb: API key found -> logging online to project {cfg.wandb_project!r}")
else:
    print("wandb: disabled (USE_WANDB = False) -> CSV logger only, results still recorded")

# Accumulated across seeds. Kept at module scope (rather than inside the loop cell) so a
# crash on the third seed does not lose the first two, and so a single seed can be re-run
# by calling run_one_seed() again.
results = []

print(f"seeds        : {SEEDS}")
print(f"max_epochs   : {MAX_EPOCHS}" + ("   (SMOKE TEST — not a real measurement)" if SMOKE_TEST else ""))
print(f"deterministic: {cfg.deterministic!r}")
print(f"train/val    : split comes from the `split` column of {os.path.basename(cfg.data.filament_index_path)}")


wandb: disabled (USE_WANDB = False) -> CSV logger only, results still recorded
seeds        : [42, 43]
max_epochs   : 5   (SMOKE TEST — not a real measurement)
deterministic: 'warn'
train/val    : split comes from the `split` column of full_fil_data.csv


## Run one seed

Everything the reference notebook does in a straight line — dataloaders, model, weights,
LoRA, metrics, Lightning module, loggers, trainer, fit — collapsed into one function so it
can be repeated from a clean slate.

**Ordering matters here.** `L.seed_everything()` is the *first* thing the function does,
before the model exists. The reference notebook calls it after `apply_peft_lora()`, which
means the head's random initialization and the LoRA `A` matrices there are seeded by
whatever the ambient RNG state happened to be — not by `training.seed`. If we kept that
order, all three runs would share identical initial weights and this notebook would only
be measuring shuffle order.

Each run gets its own checkpoint directory and its own WandB run, so nothing is
overwritten. The model, trainer and loaders are freed at the end: three 366M-parameter
models held at once will not fit.

In [30]:
from downstream_apps.filament_kyle.datasets.filament_dataset import FilamentDataset
# This app's own metrics module (val_metrics reports binary chirality accuracy alongside
# MSE and RRSE); the Lightning module is the shared template one, as in the reference
# notebook — the two copies are byte-identical.
from downstream_apps.filament_kyle.metrics.template_metrics import FlareMetrics
from downstream_apps.template.lightning_modules.pl_simple_baseline import FlareLightningModule
from workshop_infrastructure.datasets.builders import build_helio_dataloaders
from workshop_infrastructure.models.finetune_models import HelioSpectformer1D
from workshop_infrastructure.utils import load_pretrained_weights


# --- Stability fixes, added after the first 2-epoch runs came out non-monotonic ---------
#
# Diagnosis from those runs: with batch_size=1 and a 7:1 dextral:sinistral training split,
# seven single-sample steps per epoch drag the output toward 0 and one yanks it toward 1.
# Final train_loss was 0.91 / 1.66 / 7.95 against a constant-predictor floor of 0.125 --
# i.e. the model fit the training data worse than a constant. That is oscillation, not slow
# convergence, and more epochs alone would only have produced more of it.

# Average this many single-sample batches into one gradient. batch_size stays 1 so that
# drop_last cannot discard the only sinistral validation filament (see the config comment).
# 4 is a compromise: a batch of 2 is usually two dextral samples and mixes nothing, while 8
# (the whole training set) would leave only 1 optimizer step per epoch. The optimizer-step
# count is printed below -- watch it, it is small.
ACCUMULATE_GRAD_BATCHES = 4

# Scale applied to head_unembed's initial weight. Its default init is
# uniform(+/-1/sqrt(1280)) over 1280 inputs, so against LayerNorm'd features the initial
# output has std ~1 -- against 0/1 targets that is an initial MSE around 2 (observed:
# 1.7-2.3 at step 1), and the first several steps go on finding the output scale rather than
# learning anything. Shrinking the weight to 1% starts predictions at ~0.01, i.e. right at
# the always-dextral floor of 0.125, so training can only improve on the trivial solution.
#
# Scaled rather than exactly zeroed on purpose: a zero weight also zeroes the gradient to
# everything upstream (head_linear, the CLS token, every LoRA adapter) for the first
# optimizer step, and with accumulation there are only a handful of steps in a short run.
HEAD_INIT_SCALE = 0.01


def run_one_seed(seed: int) -> dict:
    """Train the pinned config once at `seed` and return its validation numbers."""
    print(f"\n{'=' * 70}\n  seed {seed}  ({MAX_EPOCHS} epochs)\n{'=' * 70}")

    # 1. Seed FIRST — before the model is constructed, so that the head initialization and
    #    the LoRA A matrices are a function of `seed`. This is the one ordering difference
    #    from 2_finetune_template_1D.ipynb, and the experiment depends on it.
    L.seed_everything(seed, workers=True)

    # 2. Dataloaders. Passing seed= pins the shuffle order to this run rather than to
    #    cfg.seed, so shuffle variance is included in the measured spread. The datasets are
    #    rebuilt each time, which is cheap: it re-does an 11-row merge_asof, not any I/O.
    train_loader, val_loader = build_helio_dataloaders(
        cfg,
        FilamentDataset,
        scalers=scalers,
        num_workers=4,          # fewer workers than the script: notebooks start faster
        seed=seed,
        #### Downstream (DS) specific parameters
        return_surya_stack=True,
        max_number_of_samples=cfg.data.max_samples,
        filament_index_path=cfg.data.filament_index_path,
        ds_time_column=cfg.data.ds_time_column,
        ds_time_tolerance=cfg.data.ds_time_tolerance,
        ds_match_direction=cfg.data.ds_match_direction,
    )

    # 3. A fresh model every run. Reusing one would train the same weights three times.
    model = HelioSpectformer1D.from_config(
        cfg.model,
        num_outputs=1,
        dtype=cfg.dtype,
        use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
    )
    load_pretrained_weights(model, cfg.model.pretrained_path)

    # Shrink the readout so the run starts at the trivial predictor (see HEAD_INIT_SCALE).
    # This MUST happen before apply_peft_lora(): PEFT wraps head_unembed in a
    # ModulesToSaveWrapper that copies the module, and reaching through the wrapper
    # afterwards can hand back the frozen original rather than the trainable copy.
    with torch.no_grad():
        model.head_unembed.weight.mul_(HEAD_INIT_SCALE)
        model.head_unembed.bias.zero_()

    # Same three regimes as the reference notebook. Under the Tier 1 config this takes the
    # use_lora branch; freeze_backbone is a no-op there (PEFT freezes everything, then
    # re-enables the adapters and every head_* module).
    if cfg.model.freeze_backbone:
        for name, param in model.named_parameters():
            if name.startswith("backbone."):
                param.requires_grad = False
    if cfg.model.use_lora:
        model = apply_peft_lora(model, cfg.model.lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[seed {seed}] trainable parameters: {trainable:,}")

    # 4. Metrics and Lightning module. Rebuilt per run so the cached torchmetrics
    #    instances do not carry state across seeds.
    metrics = {
        "train_loss": FlareMetrics("train_loss"),
        "val_loss": FlareMetrics("val_loss"),
        "train_metrics": FlareMetrics("train_metrics"),
        "val_metrics": FlareMetrics("val_metrics"),
    }
    lit_model = FlareLightningModule(
        model, metrics, lr=cfg.learning_rate, batch_size=cfg.batch_size
    )

    # 5. One logger set and one checkpoint directory per seed, so runs cannot overwrite
    #    each other's best checkpoint.
    run_name = f"noise_floor_seed{seed}"
    loggers = [CSVLogger("runs", name=run_name)]
    if USE_WANDB:
        loggers.append(
            WandbLogger(
                entity=cfg.wandb_entity,
                project=cfg.wandb_project,
                name=run_name,
                log_model=False,
                save_dir="./wandb/wandb_tmp",
            )
        )

    checkpoint_cb = ModelCheckpoint(
        dirpath=Path("checkpoints/noise_floor") / run_name,
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    # Gradient accumulation makes optimizer steps scarce, so say how many there will be.
    # This is the number that governs whether the run can converge at all.
    opt_steps_per_epoch = max(1, len(train_loader) // ACCUMULATE_GRAD_BATCHES)
    print(f"[seed {seed}] {len(train_loader)} batches/epoch / accum {ACCUMULATE_GRAD_BATCHES}"
          f" = {opt_steps_per_epoch} optimizer step(s)/epoch,"
          f" {opt_steps_per_epoch * MAX_EPOCHS} total")

    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        precision="bf16-mixed",
        logger=loggers,
        callbacks=[checkpoint_cb],
        # Passed through from training.deterministic ("warn"). Without this the config key
        # has no effect and two identical runs can differ, which would make the spread this
        # notebook reports a mixture of seed effects and nondeterminism.
        deterministic=cfg.deterministic,
        # Average ACCUMULATE_GRAD_BATCHES single-sample gradients into one optimizer step, so
        # the 7:1 class imbalance is reflected in one averaged gradient instead of several
        # conflicting ones. See the comment on the constant above.
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
        # 1, not 2: log_every_n_steps counts optimizer steps, and accumulation makes those
        # scarce (2 per epoch here). At 2 the trajectory would be nearly invisible, and
        # seeing whether train_loss descends monotonically is the whole point of this run.
        log_every_n_steps=1,
    )
    trainer.fit(lit_model, train_loader, val_loader)

    # 6. Collect. best_val_loss is what ModelCheckpoint selected on; last_val_loss is the
    #    end-of-training value. Both are worth keeping: with 3 validation samples the best
    #    epoch is easily a fluke, and a large best-vs-last gap is itself a warning.
    m = trainer.callback_metrics

    def scalar(key):
        v = m.get(key)
        return float(v) if v is not None else float("nan")

    best = checkpoint_cb.best_model_score
    result = {
        "seed": seed,
        "epochs": MAX_EPOCHS,
        "accum": ACCUMULATE_GRAD_BATCHES,
        "trainable": trainable,
        "best_val_loss": float(best) if best is not None else float("nan"),
        "last_val_loss": scalar("val_loss"),
        "last_val_mse": scalar("val_metric_mse"),
        "last_val_rrse": scalar("val_metric_rrse"),
        "last_val_accuracy": scalar("val_metric_accuracy"),
        "last_train_loss": scalar("train_loss"),
    }
    print(f"[seed {seed}] best val_loss = {result['best_val_loss']:.6f} | "
          f"last val_loss = {result['last_val_loss']:.6f} | "
          f"last val accuracy = {result['last_val_accuracy']:.4f}")

    # 7. Free the GPU before the next seed builds its own 366M-parameter copy.
    if USE_WANDB:
        wandb.finish()
    del trainer, lit_model, model, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


## Run the three seeds

Sequential, not parallel: each run wants the whole GPU. The partial summary is written to
`runs/noise_floor_summary.csv` after every seed, so an interrupted sweep keeps whatever
finished. Re-running this cell appends more runs rather than replacing them — clear
`results` first if that is not what you want.

In [31]:
SUMMARY_CSV = Path("runs/noise_floor_summary.csv")
SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    results.append(run_one_seed(seed))
    # Written after every seed rather than at the end: these runs are long enough that
    # losing two completed ones to a failure in the third would hurt.
    pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)

print(f"\n{len(results)} run(s) complete -> {SUMMARY_CSV}")


Seed set to 42



  seed 42  (5 epochs)


Loading pretrained weights from /home/jovyan/surya_workshop/downstream_apps/filament_kyle/assets/surya.366m.v1.pt.


Using bfloat16 Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


...

/home/jovyan/envs/surya_ws/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type      | Params | Mode  | FLOPs
----------------------------------------------------
0 | model | PeftModel | 362 M  | train | 0    
----------------------------------------------------
3.2 M     Trainable params
359 M     Non-trainable params
362 M     Total params
1,449.868 Total estimated model params size (MB)
540       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...
Traceback (most recent call last):
  File "<string>", line 1, in <module>


  File "/home/jovyan/envs/surya_ws/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/envs/surya_ws/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/surya_workshop/downstream_apps/filament_kyle/../../downstream_apps/filament_kyle/datasets/filament_dataset.py", line 4, in <module>
    from workshop_infrastructure.datasets.helio import HelioNetCDFDataset
  File "/home/jovyan/surya_workshop/downstream_apps/filament_kyle/../../workshop_infrastructure/datasets/helio.py", line 34, in <module>
    from workshop_infrastructure.utils import (
  File "/home/jovyan/surya_workshop/downstream_apps/filament_kyle/../../workshop_infrastructure/utils.py", line 12, in <module>
    import lightning as L
  File "/home/jovyan/envs/surya_ws/lib/python3.12/site-packag

AttributeError: 'tuple' object has no attribute 'tb_frame'

## The noise floor

`spread = max - min` over the three `best_val_loss` values. That is the number to carry
into every later comparison: **a config that beats another by less than this spread has not
been shown to be better.** Standard deviation over three runs is reported too, but with
n=3 the spread is the more honest summary.

Two sanity checks before you trust it:

- **Prior collapse.** If all three `best_val_loss` values sit near **0.2656**, every run
  learned the 7:1 training prior and nothing else. The spread is then a measurement of
  noise around a degenerate solution, which is still a valid floor but tells you the task
  is not being learned at this sample size.
- **Accuracy quantisation.** With 3 validation filaments, `last_val_accuracy` can only be
  0, 0.3333, 0.6667 or 1.0. A constant dextral predictor scores 0.6667. Do not read a
  difference between 0.6667 and 0.6667 as agreement on anything.

In [23]:
df = pd.DataFrame(results).set_index("seed")
print(df.to_string(float_format=lambda x: f"{x:.6f}"))

s = df["best_val_loss"]
spread = s.max() - s.min()

print(f"\nbest_val_loss over {len(s)} seed(s):")
print(f"  min    {s.min():.6f}")
print(f"  max    {s.max():.6f}")
print(f"  mean   {s.mean():.6f}")
print(f"  std    {s.std(ddof=1):.6f}" if len(s) > 1 else "  std    n/a (single seed)")
print(f"\n  NOISE FLOOR (max - min) = {spread:.6f}"
      + ("   <-- not a spread with one seed; run >= 2" if len(s) < 2 else ""))

# --- Trivial-predictor reference, computed from the catalog rather than hardcoded ---------
# A model that learns only the training class balance predicts the constant
# p = mean(train labels) for every input. Its val MSE is the floor below which nothing has
# been learned about chirality. This is recomputed from the catalog on every run: the
# previous hardcoded 0.2656 was correct only for the 8-train/7:1 split, so growing the event
# list would have left a silently stale reference here.
#
# Caveat: these labels come straight from the catalog, whereas the datasets additionally drop
# any event with no Surya frame inside data.ds_time_tolerance. The two agree only when every
# event matches -- compare the train/val counts below against the dataloader cell's output.
cat = pd.read_csv(cfg.data.filament_index_path)
if "split" in cat.columns:
    train_y = cat.loc[cat["split"] == "train", "chirality"].astype(float)
    val_y = cat.loc[cat["split"] == "val", "chirality"].astype(float)
else:
    train_y = val_y = cat["chirality"].astype(float)
    print("\n  NOTE: catalog has no `split` column; using all events for both references.")

prior = float(train_y.mean())
prior_collapse_val_mse = float(((val_y - prior) ** 2).mean())
constant_zero_val_mse = float((val_y ** 2).mean())
majority_val_acc = float(max((val_y == 0).mean(), (val_y == 1).mean()))

print(f"\ntrivial-predictor reference ({len(cat)} catalog events,"
      f" {len(train_y)} train / {len(val_y)} val):")
print(f"  training class prior            p = {prior:.4f}")
print(f"  val MSE predicting only p         = {prior_collapse_val_mse:.4f}   <-- collapse floor")
print(f"  val MSE predicting constant 0     = {constant_zero_val_mse:.4f}")
print(f"  val accuracy of majority class    = {majority_val_acc:.4f}")

# An absolute tolerance, not just the spread: with a single seed the spread is 0, so the
# original `abs(mean - floor) < spread` test could never fire. That is exactly what happened
# on the first 5-epoch run -- best_val_loss 0.2586 against a 0.2656 floor, entirely collapsed
# and silently unflagged. max(spread, PRIOR_TOL) keeps the multi-seed behaviour and makes the
# check work at n=1.
PRIOR_TOL = 0.02
tol = max(spread, PRIOR_TOL)
mean_best = float(s.mean())

if abs(mean_best - prior_collapse_val_mse) < tol:
    print(f"\n  WARNING: mean best_val_loss ({mean_best:.4f}) is within {tol:.4f} of the")
    print(f"  prior-collapse floor ({prior_collapse_val_mse:.4f}). These runs most likely")
    print("  learned the training class balance and nothing about chirality.")
    print(f"  Confirm it: if last_val_accuracy above equals {majority_val_acc:.4f}"
          " (the majority-class score),")
    print("  the model is predicting one class for every input.")
elif mean_best < prior_collapse_val_mse - tol:
    print(f"\n  mean best_val_loss ({mean_best:.4f}) is below the prior-collapse floor"
          f" ({prior_collapse_val_mse:.4f})")
    print(f"  by more than {tol:.4f} -- something beyond the class balance was learned.")
else:
    print(f"\n  WARNING: mean best_val_loss ({mean_best:.4f}) is WORSE than the prior-collapse")
    print(f"  floor ({prior_collapse_val_mse:.4f}) -- worse than predicting a single constant.")
    print("  Suspect an unstable run rather than a hard task; check the per-step trajectory in")
    print("  runs/noise_floor_seed*/version_*/metrics.csv.")

if SMOKE_TEST:
    print("\n  SMOKE_TEST is True -- this is a short dry run, NOT the noise floor.")
    print("  Set SMOKE_TEST = False, restart, and re-run for the real measurement.")


      epochs  accum  trainable  best_val_loss  last_val_loss  last_val_mse  last_val_rrse  last_val_accuracy  last_train_loss
seed                                                                                                                         
42         5      4    3157761       0.258633       0.263579      0.263579    1054.060425           0.666667         0.009922

best_val_loss over 1 seed(s):
  min    0.258633
  max    0.258633
  mean   0.258633
  std    n/a (single seed)

  NOISE FLOOR (max - min) = 0.000000   <-- not a spread with one seed; run >= 2

trivial-predictor reference (11 catalog events, 8 train / 3 val):
  training class prior            p = 0.1250
  val MSE predicting only p         = 0.2656   <-- collapse floor
  val MSE predicting constant 0     = 0.3333
  val accuracy of majority class    = 0.6667

  prior-collapse floor (0.2656). These runs most likely
  learned the training class balance and nothing about chirality.
  Confirm it: if last_val_accuracy abo

## Conclusion

You now have a threshold. When you run the Tier 2-4 parameter sets (regime, pooling, head
capacity), compare each `best_val_loss` against the *spread* recorded here, not against
the other configs' point estimates.

If the spread came out large relative to the differences you care about — which is a real
possibility with 8 training and 3 validation samples — that is itself the result, and the
honest response is more labelled filaments or leave-one-out cross-validation over all 11
events rather than a bigger hyperparameter sweep.